In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import json
import time

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
# json, table (markdown), yaml, xml

In [4]:
plain_prompt = """다음 제품 리뷰를 분석해줘. 전체 감정(긍정/부정/혼합), 1~5점 점수, 장점 목록, 단점 목록, 핵심 키워드 3개를 알려줘.
리뷰 : "이 노트북 정말 가벼워서 좋아요! 다만 키보드 타건감이 아쉽네요"
유효한 json만 출력하세요"""

print(llm.invoke([HumanMessage(content=plain_prompt)]).content)

```json
{
  "overall_sentiment": "혼합",
  "score": 4,
  "pros": ["가벼운 무게"],
  "cons": ["타건감 아쉬움"],
  "keywords": ["가벼움", "키보드", "타건감"]
}
```


# 헤더1입니다.

## 헤더2입니다.

### 헤더3입니다.

**abcde**

abcde

- a1
- a2
- a3

In [3]:
markdown_prompt = """# 제품 리뷰 감정 분석

## 입력 리뷰
> "이 노트북 정말 가벼워서 좋아요! 다만 키보드 타건감이 아쉽네요"

## 분석 항목
- **overall_sentiment** : 긍정/부정/혼합 중 하나
- **score** : 1~5점수
- **pros** : 장점 리스트
- **cons** : 단점 리스트
- **keywords** : 핵심 키워드 3개

## 출력 형식
유효한 json만 출력하세요. 다른 텍스트를 포함하지 마세요
"""

print(llm.invoke([HumanMessage(content=markdown_prompt)]).content)

{
  "overall_sentiment": "혼합",
  "score": 3,
  "pros": [
    "가벼움"
  ],
  "cons": [
    "키보드 타건감"
  ],
  "keywords": [
    "노트북",
    "가벼움",
    "키보드"
  ]
}


```json
{
    "overall_sentiment": "혼합",
    "score": 3,
    "pros": ["가벼움"],
    "cons": ["키보드 타건감"],
    "keywords": ["노트북", "가벼워서", "키보드"]
}
```

In [6]:
json_prompt = """다음 제품 리뷰를 분석해서 json 형식으로 출력하세요

리뷰 : "이 노트북 정말 가벼워서 좋아요! 다만 키보드 타건감이 아쉽네요"

출력 형식
{
    "overall_sentiment" : 긍정/부정/혼합,
    "score" : 1~5점수,
    "pros" : [장점],
    "cons" : [단점],
    "keywords" : [키워드1, 키워드2, 키워드3]
}
"""

print(llm.invoke([HumanMessage(content=json_prompt)]).content)

```json
{
    "overall_sentiment": "혼합",
    "score": 3,
    "pros": ["가벼움"],
    "cons": ["키보드 타건감"],
    "keywords": ["노트북", "가벼워서", "키보드"]
}
```


In [ ]:
# Parser

In [ ]:
# 회의록 -> 구조화된 데이터로 변환
json_structure = {
    'date' : 'YYYY-MM-DD',
    'attendees' : ['이름1', '이름2'],
    'agenda' : ['안건1', '안건2'],
    'decisions' : ['결정1'],
    'action_items' : [{'assignee' : '담당자', 'task' : '작업', 'deadline' : '기한'}]
}

In [10]:
text = '3월 15일 마케팅팀 주간 회의. 참석: 김팀장, 이대리, 박사원. 신규 SNS 캠페인 예산 5000만원 확정. 이대리가 3월 22일까지 시안 준비. 박사원은 경쟁사 분석 보고서 3월 20일까지.'

def generate_meeting_minute(raw_text):
    messages = [
        SystemMessage(content = '당신은 회의록 전문가입니다. 인사말이나 부연설명 없이 순수한 json 데이터만 반환하세요'),
        HumanMessage(content = f"""다음 회의 내용을 지정된 json 구조로 변환하세요.
        
        회의내용 : {raw_text}
        
        json구조:
        {{
            'date' : 'YYYY-MM-DD',
            'title' : '회의 제목',
            'attendees' : ['이름1', '이름2'],
            'agenda' : ['안건1', '안건2'],
            'decisions' : ['결정1'],
            'action_items' : [
                    {{'assignee' : '담당자', 'task' : '작업', 'deadline' : '기한'}}]
        }}
        """)
    ]

    response = llm.invoke(messages).content
    
    return response
    

In [12]:
print(generate_meeting_minute(text))

{
    "date": "2023-03-15",
    "title": "마케팅팀 주간 회의",
    "attendees": ["김팀장", "이대리", "박사원"],
    "agenda": ["신규 SNS 캠페인 예산 확정", "경쟁사 분석 보고서"],
    "decisions": ["신규 SNS 캠페인 예산 5000만원 확정"],
    "action_items": [
        {"assignee": "이대리", "task": "시안 준비", "deadline": "2023-03-22"},
        {"assignee": "박사원", "task": "경쟁사 분석 보고서", "deadline": "2023-03-20"}
    ]
}


In [ ]:
# prompt : 제약조건
# 길이 : 3문장으로 요약해서 말하세요, 100자 이내로 얘기하세요
# 내용 : 검색된 정보로만 얘기하세요, 주어진 정보 안에서만 얘기하세요, 추측하지 마세요
# 형식 : json 형태로만 출력하세요, 불릿포인트만 사용해서 출력하세요
# 부정제약 : 거짓말로 지어내지마세요 -> 사실만 그대로 얘기하세요

In [25]:
class ConstrainedPrompt:
    def __init__(self, base_instruction):
        self.instruction = base_instruction
        self.constraints = []
    
    def add_length(self, description):
        self.constraints.append(f"[길이] {description}")
        return self
    
    def add_content(self, description):
        self.constraints.append(f"[내용] {description}")
        return self
    
    def add_format(self, description):
        self.constraints.append(f"[형식] {description}")
        return self
        
    def add_style(self, description):
        self.constraints.append(f"[스타일] {description}")
        return self
        
    def build(self):
        parts = [self.instruction, "\n제약조건:"]
        for c in self.constraints:
            parts.append(f" - {c}")
        return '\n'.join(parts)
    
    def execute(self, **kwargs):   #  # execute()
#         print('kwargs: ', kwargs)  # kwargs['temperature'], kwargs['max_tokens']
        prompt = self.build()
#         llm = ChatOpenAI(model="gpt-4o-mini", temperature = kwargs['temperature'], max_tokens = kwargs['max_tokens'])
        return llm.invoke([HumanMessage(content=prompt)]).content

In [26]:
result = (
    ConstrainedPrompt('클라우드 컴퓨팅의 장점을 설명해주세요')
    .add_length("5개의 불릿포인트")
    .add_content('비용, 확장성, 보안 관점을 반드시 포함')
    .add_format('각 포인트는 한줄로, 이모지로 시작')
    .add_style('IT 비전공 경영진을 대상, 전문 용어에 괄호로 설명 추가')
    .execute(temperature=0.3, max_tokens=200)

)

kwargs:  {'temperature': 0.3, 'max_tokens': 200}


In [18]:
print(result)

- 💰 **비용 절감**: 클라우드 서비스는 초기 하드웨어 비용을 줄여주고, 사용한 만큼만 비용을 지불하는 체계로 경비를 효율적으로 관리할 수 있습니다.  
- 📈 **확장성**: 비즈니스 성장에 맞춰 신속하게 리소스(자원)를 추가할 수 있어 수요 변화에 유연하게 대응할 수 있습니다.  
- 🔒 **보안 강화**: 클라우드 제공업체는 통합된 보안 솔루션을 통해 데이터 보호 및 관리에 최선을 다하며, 최신 기술을 지속적으로 업데이트합니다.  
- 🌐 **접근성 향상**: 인터넷만 있으면 언제 어디서나 데이터에 접근할 수 있어 팀의 협업과 의사소통이 용이해집니다.  
- ⚙️ **자동화 및 관리 용이**: 클라우드 플랫폼은 자동 업데이트 및 관리 도구를 제공하여 IT 관리의 부담을 덜어줍니다.  


In [ ]:
# ConstrainedPrompt
# 광고카피
# 상세설명

In [27]:
product = "에어프로 맥스 무선 헤드폰"
features = ["40시간 배터리", "멀티포인트 연결", "30dB 노이즈캔슬링", "300g 경량 설계"]

In [34]:
ad_result = (
    ConstrainedPrompt(f"{product} 제품 광고 카피를 작성해주세요. 특징 : {', '.join(features)}")
    .add_length("한 줄, 30자 이내")
    .add_content('핵심 특징 1가지만 강조')
    .add_format('슬로건 형태, 마침표 없이')
    .add_style('2030 타겟, 감성적이고 트렌디한 톤')
    .execute()
)

kwargs:  {}


In [31]:
detail_result = (
    ConstrainedPrompt(f"{product} 상세 제품 설명을 작성해주세요. 특징 : {', '.join(features)}")
    .add_length("200자 내외, 3단락")
    .add_content('모든 특징을 빠짐없이 설명, 사용 시나리오 1개 포함')
    .add_format('단락 구분을 명확히, 마지막에 가격 정보 위치 표시')
    .add_style('공식 제품 페이지 톤, 신뢰감 있는 문체')
    .execute()
)

kwargs:  {}


In [35]:
print(ad_result)

끝없는 음악과 함께하는 40시간의 자유


In [33]:
print(detail_result)

에어프로 맥스 무선 헤드폰은 탁월한 성능과 편안한 착용감을 제공합니다. 40시간의 배터리 수명으로, 장시간 음악 감상이나 회의 시에도 걱정 없이 사용 가능합니다. 멀티포인트 연결 기능을 통해 두 개의 기기를 동시에 연결하여 스마트폰과 노트북을 손쉽게 전환할 수 있습니다.

또한, 30dB의 뛰어난 노이즈 캔슬링 기능이 탑재되어 있어 시끄러운 환경에서도 몰입할 수 있는 경험을 제공합니다. 300g의 가벼운 설계로 편안한 착용감이 유지되어, 장시간 사용에도 부담이 없습니다. 출퇴근길이나 카페에서의 집중 시간에 최적화된 성능을 발휘합니다.

이 제품은 가격이 249,000원입니다. 뛰어난 기능과 디자인을 갖춘 에어프로 맥스는 당신의 일상에 편리함과 즐거움을 더할 것입니다.


In [36]:
company_policy = """
[모두컴퍼니 재택근무 정책 v2.3]
- 주 3일 재택, 2일 출근 (화/목 필수 출근)
- 재택근무 시 오전 9시까지 Slack 상태 '업무중' 설정 필수
- 해외 원격근무는 최대 연속 2주까지 가능 (사전 승인 필요)
- 야간근무(22시 이후) 시 익일 오후 출근 가능
- 재택근무 장비 지원금: 연 100만원 (영수증 제출)
"""


In [37]:
question = '해외에서 한 달 동안 원격 근무할 수 있나요?'

In [38]:
with_context = f"""아래 회사 정책 문서를 참고하여 질문에 답하세요.
문서에 없는 내용은 "해당 정책 문서에 명시되어 있지 않습니다"라고 답하세요.

정책 문서:
\"\"\"{company_policy}\"\"\"

질문 : {question}"""

print(llm.invoke([HumanMessage(content=with_context)]).content)

해당 정책 문서에 명시되어 있지 않습니다.


In [48]:
def answer_with_context(context, question):
    clause = """
    중요 : 반드시 제공된 문서 내용만을 근거로 답변하세요.
    문서에 없는 내용은 "제공된 문서에 해당 정보가 없습니다"라고 답하세요.
    추측하거나 외부 지식을 사용하지 마세요."""
    
    prompt = f"""아래 참고 문서를 기반으로 질문에 답하세요.
    {clause}
    
    참고 문서:
    \"\"\"{company_policy}\"\"\"
    
    질문 : {question}
    
    답변 형식:
    - 답변 : [핵심 답변]
    - 근거 : [문서에서 관련 부분 인용]"""
    
    return llm.invoke([HumanMessage(content=prompt)]).content

In [47]:
answer_with_context(company_policy, question)

'- 답변 : 제공된 문서에 해당 정보가 없습니다.'

In [50]:
answer_with_context(company_policy, '식대 지원금은 얼마인가요?')

'- 답변 : 제공된 문서에 해당 정보가 없습니다.\n- 근거 : 제공된 문서에는 식대 지원금에 대한 내용이 포함되어 있지 않습니다.'

In [51]:
# zero-shot, few-shot

In [ ]:
# 식대 지원금에 대한 내용이 포함되어 있지 않습니다 -> 감정 분석해주세요

# 식대 지원금에 대한 내용이 포함되어 있지 않습니다 : 긍정 -> 제공된 문서에 해당 정보가 없습니다. 감정 분석해주세요

In [52]:
categories = ["기술", "경제", "스포츠", "문화", "정치"]

news_articles = [
    "삼성전자가 차세대 AI 반도체 개발에 3조원을 투자한다고 발표했다.",
    "한국은행이 기준금리를 0.25%p 인하하며 경기 부양에 나섰다.",
    "손흥민이 프리미어리그 시즌 최다 도움을 기록하며 팀 승리를 이끌었다.",
    "국립현대미술관에서 한국 현대미술 50년 특별전이 개막했다."
]

In [53]:
classify_prompt = f"""당신은 뉴스 분류 전문가입니다.
주어진 뉴스 기사를 다음 카테고리 중 하나로 분류하세요: {', '.join(categories)}
반드시 카테고리 이름만 출력하세요"""

for article in news_articles:
    result = llm.invoke([
        SystemMessage(content = classify_prompt),
        HumanMessage(content = article)
    ]).content
    print(f" 기사 : {article[:30]}...")
    print(f" 분류 : {result}\n")

 기사 : 삼성전자가 차세대 AI 반도체 개발에 3조원을 투자한다...
 분류 : 기술

 기사 : 한국은행이 기준금리를 0.25%p 인하하며 경기 부양에...
 분류 : 경제

 기사 : 손흥민이 프리미어리그 시즌 최다 도움을 기록하며 팀 승...
 분류 : 스포츠

 기사 : 국립현대미술관에서 한국 현대미술 50년 특별전이 개막했...
 분류 : 문화



In [ ]:
# long_text = "~~~~"

# for article in news_articles:
#     result = llm.invoke([
#         SystemMessage(content = '당신은 요약 전문가입니다~~'),
#         HumanMessage(content = article)
#     ]).content
#     print(f" 기사 : {article[:30]}...")
#     print(f" 분류 : {result}\n")

In [54]:
few_shot_messages = [
    SystemMessage(content = '주어진 리뷰의 감정을 분석하세요'),
    #예시..
    HumanMessage(content = "리뷰: '이 제품 정말 최고입니다! 강력 추천해요'"),
    AIMessage(content = '{"sentiment" : "긍정", "score" : 0.95, "keywords" : ["최고", "강력 추천"]}'),
    # 예시
    HumanMessage(content = "리뷰: '배송도 느리고 제품 품질도 형편없네요'"),
    AIMessage(content = '{"sentiment" : "부정", "score" : 0.15, "keywords" : ["느리고", "형편없네요"]}'),
    #예시
    HumanMessage(content = "리뷰: '가격 대비 괜찮지만, 기대했던 것보다는 아쉬워요'"),
    AIMessage(content = '{"sentiment" : "혼합", "score" : 0.50, "keywords" : ["괜찮지만", "아쉬워요"]}'),
    
    #실제 쿼리
    HumanMessage(content = "리뷰: '디자인은 예쁜데, 베터리가 빨리 닳아요. 전체적으로 보통입니다'")
]

print(llm.invoke(few_shot_messages).content)

{"sentiment" : "혼합", "score" : 0.55, "keywords" : ["예쁜데", "빨리 닳아요", "보통"]}


In [55]:
examples = [
    {"informal": "내일 미팅 좀 미룰 수 있을까? 갑자기 일이 생겼어.",
     "formal": "안녕하세요. 내일 예정된 미팅 일정 변경을 요청드립니다. 긴급한 업무가 발생하여 조율이 필요합니다. 가능한 대체 일정을 알려주시면 감사하겠습니다."},
    {"informal": "그 보고서 다 했어? 빨리 보내줘.",
     "formal": "안녕하세요. 요청드렸던 보고서 진행 상황을 확인드립니다. 완료되셨다면 전달 부탁드리며, 추가 시간이 필요하시면 말씀해 주세요."},
    {"informal": "이번 프로젝트 결과 별로인데 어떻게 할까?",
     "formal": "안녕하세요. 이번 프로젝트 결과에 대해 논의가 필요합니다. 개선 방안을 함께 검토하기 위해 미팅을 잡는 것이 어떨까요?"}
]

In [56]:
test_messages = [
    "다음 주 워크숍 참석 못 할 것 같아. 다른 사람 보내도 돼?",
    "예산 좀 더 받을 수 있을까? 지금 부족해."
]

In [60]:
messages = [SystemMessage(content = '당신은 비즈니스 커뮤니케이션 전문가입니다. 비공식적인 메시지를 정중한 비즈니스 이메일 톤으로 변환하세요. 인사말, 존칭, 구체적 표현을 포함하세요')]
for ex in examples:
    messages.append(HumanMessage(content = ex['informal']))
    messages.append(AIMessage(content = ex['formal']))
    
messages.append(HumanMessage(content = test_messages[0] ))
messages

[SystemMessage(content='당신은 비즈니스 커뮤니케이션 전문가입니다. 비공식적인 메시지를 정중한 비즈니스 이메일 톤으로 변환하세요. 인사말, 존칭, 구체적 표현을 포함하세요', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='내일 미팅 좀 미룰 수 있을까? 갑자기 일이 생겼어.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요. 내일 예정된 미팅 일정 변경을 요청드립니다. 긴급한 업무가 발생하여 조율이 필요합니다. 가능한 대체 일정을 알려주시면 감사하겠습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='그 보고서 다 했어? 빨리 보내줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요. 요청드렸던 보고서 진행 상황을 확인드립니다. 완료되셨다면 전달 부탁드리며, 추가 시간이 필요하시면 말씀해 주세요.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='이번 프로젝트 결과 별로인데 어떻게 할까?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요. 이번 프로젝트 결과에 대해 논의가 필요합니다. 개선 방안을 함께 검토하기 위해 미팅을 잡는 것이 어떨까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='다음 주 

In [61]:
llm.invoke(messages).content

'안녕하세요. 다음 주 워크숍 참석이 어려울 것 같아 이렇게 말씀드립니다. 대리 참석 가능성을 검토해 주시면 감사하겠습니다. 다른 참석자를 선정하실 경우, 알려주시면 도움이 될 것 같습니다. 늘 감사합니다.'

In [62]:
messages = [SystemMessage(content = '당신은 비즈니스 커뮤니케이션 전문가입니다. 비공식적인 메시지를 정중한 비즈니스 이메일 톤으로 변환하세요. 인사말, 존칭, 구체적 표현을 포함하세요')]
for ex in examples:
    messages.append(HumanMessage(content = ex['informal']))
    messages.append(AIMessage(content = ex['formal']))
    
messages.append(HumanMessage(content = test_messages[1] ))
llm.invoke(messages).content

'안녕하세요. 현재 진행 중인 프로젝트 예산에 관해 논의드리고자 합니다. 현재 예산이 부족하여 추가 지원이 필요한 상황입니다. 가능할지 검토해 주시면 감사하겠습니다.'